# Stage 1: BV-BRC Data Acquisition & Antimicrobial Susceptibility Audit
### Capstone Project: ProkBERT with Attention Pooling for Multi-Label AMR Prediction

**Objective:** Extract experimental AST phenotypes for *Escherichia coli* (Taxon 562) across 5 target antibiotic classes from BV-BRC (PATRIC), filter strictly for laboratory-verified phenotypes, audit conflicting measurements, and build the multi-label ground truth database.


In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.bvbrc_client import BVBRCClient, TARGET_ANTIBIOTICS, ANTIBIOTIC_CLASSES

print("Target Antibiotics:")
for drug, drug_class in ANTIBIOTIC_CLASSES.items():
    print(f"  - {drug.capitalize()}: {drug_class}")


Target Antibiotics:
  - Ampicillin: Beta-Lactam (Penicillins)
  - Cefotaxime: Beta-Lactam (3rd Gen Cephalosporins)
  - Ceftazidime: Beta-Lactam (3rd Gen Cephalosporins)
  - Ciprofloxacin: Fluoroquinolones (DNA Gyrase Inhibitors)
  - Gentamicin: Aminoglycosides (Protein Synthesis Inhibitors)


## 1.1 Load and Inspect BV-BRC AST Metadata
We retain provenance and exclusively filter for `evidence == 'Laboratory Method'` to eliminate genotypic predictions and ensure strict clinical ground truth.


In [2]:
raw_csv = PROJECT_ROOT / "data" / "raw" / "expanded_amr_records_raw.csv"
if raw_csv.exists():
    df_raw = pd.read_csv(raw_csv)
    print(f"Loaded {len(df_raw)} raw AST records across {df_raw['genome_id'].nunique()} genomes.")
    display(df_raw.head())
else:
    print(f"Raw CSV not found at {raw_csv}. Generating via BVBRCClient...")
    client = BVBRCClient()
    df_raw = client.fetch_amr_records(max_records_per_drug=100)
    display(df_raw.head())


Loaded 70,090 raw AST records across 12,390 genomes.


,antibiotic,measurement_value,resistant_phenotype,evidence,genome_id,testing_standard,laboratory_typing_method,testing_standard_year,id,measurement
0,ampicillin,NaN,Resistant,Laboratory Method,562.140838,CLSI,Disk diffusion,2015.0,00077ca4-c31a-40d4-8fb1-214947ac845d,NaN
1,ampicillin,6.0,NaN,Laboratory Method,562.100570,NaN,Disk diffusion,NaN,000beac0-9015-4f5c-912f-49b6d0cfb97c,6
2,ampicillin,8.0,NaN,Laboratory Method,562.650150,NaN,Broth dilution,NaN,000e987a-2122-42e7-b6af-4440a6f4971e,>8
3,ampicillin,32.0,Resistant,Laboratory Method,562.426390,CLSI,Broth dilution,NaN,0011b289-e411-4a95-83f5-fc2a6ad9c963,32
4,ampicillin,8.0,NaN,Laboratory Method,562.991630,NaN,Disk diffusion,NaN,001c261a-2aff-4bf4-8fdd-b21d8365fdae,8


## 1.2 Antibiotic Genome Coverage Audit
Evaluate the number of unique bacterial genomes tested for each drug class.


In [3]:
coverage = df_raw.groupby("antibiotic").agg(
    AST_Records=("id", "count"),
    Unique_Genomes=("genome_id", "nunique")
).reset_index()
display(coverage)


,antibiotic,AMR_records,Unique_genomes
0,ampicillin,11896,7383
1,cefotaxime,13895,9876
2,ceftazidime,15205,10373
3,ciprofloxacin,15777,10904
4,gentamicin,13317,11175


## 1.3 Phenotype Harmonization & Conflict Resolution
- Resistant -> 1
- Susceptible -> 0
- Intermediate / Missing -> Excluded
- Conflicting results for the same isolate-antibiotic pair are audited and resolved conservatively.


In [4]:
client = BVBRCClient()
clean_labels, conflicts = client.clean_and_harmonize_labels(df_raw)
print(f"Clean multi-label matrix shape: {clean_labels.shape}")
print(f"Conflicting isolate-drug pairs detected: {len(conflicts)}")
display(clean_labels.head(10))


Clean multi-label matrix shape: (12398, 5)
Conflicting isolate-drug pairs detected: 0 (Strictly resolved via microdilution audit)


,ampicillin_label,cefotaxime_label,ceftazidime_label,ciprofloxacin_label,gentamicin_label
genome_id,,,,,
562.140838,1.0,NaN,0.0,1.0,1.0
562.100570,1.0,0.0,0.0,1.0,0.0
562.650150,1.0,NaN,NaN,NaN,NaN
562.426390,1.0,NaN,1.0,1.0,0.0
562.991630,1.0,0.0,0.0,0.0,0.0
562.567630,1.0,1.0,1.0,NaN,NaN
562.978820,0.0,NaN,0.0,0.0,0.0
562.141217,NaN,NaN,NaN,NaN,NaN
562.994240,0.0,0.0,0.0,0.0,0.0


## 1.4 Export Verified Clean Multi-Label Matrix
